# DREAM AI — train our native model from random initialization

This notebook trains DREAM's own PyTorch transformer. It does **not** download or call another model, use a third-party AI API, or import pretrained weights. The checkpoint starts with random weights.

**Important:** the current architecture is an early research-scale model, not a capable general assistant. Training requires a substantial, high-quality dataset that you have permission to use. Do not mark rights/privacy/quality review as complete unless a person has actually completed those checks. Free GPU availability and session duration are not guaranteed.

In [ ]:
from pathlib import Path
import os, subprocess, sys, torch

REPO = Path('/content/dream-ai')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/mjnr0477/dream-ai.git', str(REPO)], check=True)
os.chdir(REPO)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. You can run a tiny smoke test on CPU, but substantial training will be slow.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/DREAM_AI')
CORPUS_FILE = DRIVE_DIR / 'approved-records.jsonl'
CHECKPOINT = DRIVE_DIR / 'checkpoints' / 'dream-small.pt'
REPORT = DRIVE_DIR / 'eval-results' / 'dream-small.json'
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)
print('Persistent DREAM workspace:', DRIVE_DIR)

## 1. Upload the reviewed DREAM corpus

Prepare a JSONL file named `approved-records.jsonl`. Each line must contain `id`, `source`, `source_group`, `license`, `rights_confirmed`, `privacy_reviewed`, `quality_reviewed`, `language`, and `text`. The three review fields must be `true` only after real human review. Use at least three independent source groups. Keep personal or confidential information out of the corpus. The notebook stores the approved file in your Google Drive; delete or replace that Drive file if you intend to use a different corpus.

In [ ]:
from google.colab import files
if not CORPUS_FILE.exists():
    uploaded = files.upload()
    if 'approved-records.jsonl' not in uploaded:
        raise RuntimeError('Please upload a file named approved-records.jsonl')
    CORPUS_FILE.write_bytes(uploaded['approved-records.jsonl'])
else:
    print('Using reviewed corpus already saved in Google Drive:', CORPUS_FILE)
Path('data').mkdir(exist_ok=True)
Path('data/approved-records.jsonl').write_bytes(CORPUS_FILE.read_bytes())
print('Corpus bytes:', CORPUS_FILE.stat().st_size)

In [ ]:
subprocess.run([sys.executable, 'model/prepare_corpus.py', '--input', 'data/approved-records.jsonl', '--output', 'data/prepared'], check=True)
import json
manifest = json.loads(Path('data/prepared/manifest.json').read_text())
print(json.dumps(manifest, indent=2))
for split in ('train', 'validation', 'test'):
    size = Path(f'data/prepared/{split}.txt').stat().st_size
    print(split, 'UTF-8 bytes:', size)
    if size < 258:
        raise RuntimeError(f'{split} split is too small for the default 256-byte context. Add more reviewed data before training.')

## 2. Train DREAM from scratch

The starter settings use a small micro-batch plus gradient accumulation to reduce peak memory. Increase steps only if the runtime can finish. Training saves resumable checkpoints in the Colab session; download them before the session ends.

In [ ]:
TRAIN_STEPS = 2000
MICRO_BATCH = 4
ACCUMULATION = 8
train_command = [
    sys.executable, 'model/train.py',
    '--data', 'data/prepared/train.txt',
    '--validation-data', 'data/prepared/validation.txt',
    '--out', str(CHECKPOINT),
    '--steps', str(TRAIN_STEPS),
    '--batch-size', str(MICRO_BATCH),
    '--grad-accum-steps', str(ACCUMULATION),
    '--block-size', '256', '--d-model', '256', '--layers', '6', '--heads', '8',
    '--eval-every', '100', '--checkpoint-every', '250', '--eval-batches', '10'
]
if CHECKPOINT.exists():
    print('Resuming from the last Google Drive checkpoint:', CHECKPOINT)
    train_command += ['--resume', str(CHECKPOINT)]
subprocess.run(train_command, check=True)

## 3. Evaluate on the held-out test split

This reports next-byte prediction metrics and a sample generation. These metrics do not prove reasoning, factual accuracy, safety or instruction-following ability.

In [ ]:
subprocess.run([
    sys.executable, 'model/evaluate.py',
    '--checkpoint', str(CHECKPOINT),
    '--data', 'data/prepared/test.txt',
    '--batches', '50', '--batch-size', '8',
    '--out', str(REPORT)
], check=True)
report = json.loads(REPORT.read_text())
print('Bits per byte:', report['bits_per_byte'])
print('Perplexity per byte:', report['perplexity_per_byte'])
print('Sample:', report['sample_generation'])

## 4. Download the checkpoint and evaluation report

Keep both files with the dataset manifest. A checkpoint without its exact training and evaluation provenance is not a reproducible release.

In [ ]:
files.download(str(CHECKPOINT))
files.download(str(REPORT))
files.download('data/prepared/manifest.json')